In [76]:
import numpy as np
from causal_graph_comparison import *
from pathlib import Path
from climatem.synthetic_data.savar import dict_to_matrix
import numpy as np
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from causal_graph_comparison.graph_utils import binarize_array
from climatem.synthetic_data.graph_evaluation_ilija import extract_adjacency_matrix
from climatem.synthetic_data.savar import dict_to_matrix
from causal_graph_comparison.dim_reduction import get_mean_modes
from tigramite.independence_tests.parcorr import ParCorr
import tigramite

In [98]:
# ---- BEGIN CONFIG ----
seed = 1
difficulty = "easy"
n_modes = 4
n_lags = 5
n_inputs = n_modes * n_lags  # 20
n_outputs = n_modes          # 4
n_vars = n_inputs + n_outputs  # 24
tau = 5

# Your data: shape (n_samples, 24)
# First 20 columns: lagged inputs (lags 1-5, modes 1-4)
# Last 4 columns: outputs (modes 1-4) at prediction step
# Example: data_array = np.random.rand(1000, 24)  # REPLACE with your data

# Variable names for clarity
var_names = (
    [f'x{mode}_t{lag+1}' for lag in range(n_lags) for mode in range(n_modes)]
    + [f'x{mode}_t0' for mode in range(n_modes)]
)
# ---- END CONFIG ----

In [99]:
print("Var names: ", var_names)

Var names:  ['x0_t1', 'x1_t1', 'x2_t1', 'x3_t1', 'x0_t2', 'x1_t2', 'x2_t2', 'x3_t2', 'x0_t3', 'x1_t3', 'x2_t3', 'x3_t3', 'x0_t4', 'x1_t4', 'x2_t4', 'x3_t4', 'x0_t5', 'x1_t5', 'x2_t5', 'x3_t5', 'x0_t0', 'x1_t0', 'x2_t0', 'x3_t0']


In [100]:
# ---- LOAD DATA ----

savar_name = f"modes_{n_modes}-diff_{difficulty}-seed_{seed}"
data_path = OUTPUTS_DIR / Path(f"savar-{savar_name}-samples_1000-rollouts_1steps.npz")
data_array = np.load(data_path)["targets"]

params_file = DATA_DIR / f"{savar_name}_parameters.npy"
params = np.load(params_file, allow_pickle=True).item()
links_coeffs = params["links_coeffs"]

gt_adj = extract_adjacency_matrix(links_coeffs, n_modes, tau)
gt_adj = binarize_array(gt_adj)

# gt_adj2 = dict_to_matrix(links_coeffs)
# gt_adj2 = binarize_array(gt_adj2)

In [101]:
print("links_coeffs: ", links_coeffs)
print("gt_adj: \n", gt_adj)
print("data_array: ", data_array.shape)

links_coeffs:  {0: [((0, -2), 0.5)], 1: [((1, -3), 0.38)], 2: [((2, -2), 0.36)], 3: [((3, -4), 0.38)]}
gt_adj: 
 [[[0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]]

 [[1 0 0 0]
  [0 0 0 0]
  [0 0 1 0]
  [0 0 0 0]]

 [[0 0 0 0]
  [0 1 0 0]
  [0 0 0 0]
  [0 0 0 0]]

 [[0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 1]]

 [[0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]]]
data_array:  (998, 1, 400)


In [102]:
# dimensionality reduction
reduced_arr = get_mean_modes(data_array, n_modes, gaussian=True)
print(reduced_arr.shape)

(998, 1, 4)


In [103]:
# build targets and inputs

num_samples = reduced_arr.shape[0] - tau  # 998 - 5 = 993

inputs = np.zeros((num_samples, tau, n_modes))  # (993, 5, 4)
targets = np.zeros((num_samples, n_modes))              # (993, 4)

for i in range(num_samples):
    # print(reduced_arr[i : i + tau].squeeze().shape)
    inputs[i] = reduced_arr[i : i + tau].squeeze()           # shape (5, 4)
    # print(reduced_arr[i + tau].shape)
    targets[i] = reduced_arr[i + tau]           # shape (1, 4)


In [104]:
print("inputs: ", inputs.shape)
print("targets: ", targets.shape)

inputs:  (993, 5, 4)
targets:  (993, 4)


In [105]:
inputs_flat = inputs.reshape(num_samples, tau * n_modes)  # (993, 20)
data_array = np.concatenate([inputs_flat, targets], axis=1)  # (993, 24)
print(data_array.shape)

(993, 24)


In [110]:
# --- Prepare for Tigramite ---
# Create tigramite data frame
data = pp.DataFrame(data_array, var_names=var_names)

parcorr = ParCorr(significance='analytic')
pcmci = PCMCI(dataframe=data, cond_ind_test=parcorr)

link_assumptions = {}

for j in range(n_vars):
    if j >= n_inputs:
        # Outputs: allow links from inputs
        entries = {}
        for i in range(n_inputs):
            entries[(i, 0)] = '-?>'
        link_assumptions[j] = entries
    else:
        # Inputs: no parents allowed
        link_assumptions[j] = {}

# Run PCMCI with links mask
results = pcmci.run_pcmci(
    tau_max=tau,
    pc_alpha=None,
    link_assumptions=link_assumptions
)

# print(results['graph'][3])
# print(results['graph'][1].shape)

graph = results['graph']
p_matrix = results['p_matrix']
val_matrix = results['val_matrix']

links_list = []

# Loop through all possible links and lags
for source in range(n_vars):
    for target in range(n_vars):
        for lag in range(tau + 1):
            p_val = p_matrix[source, target, lag]
            if not np.isnan(p_val):
                # Store with lag converted to signed (negative) lag convention
                links_list.append({
                    'source': source,
                    'source_name': var_names[source],
                    'target': target,
                    'target_name': var_names[target],
                    'lag': -lag,  # assuming lag indexing: index 0 means lag 0, so lag = -index
                    'p_value': p_val,
                    'effect': val_matrix[source, target, lag]
                })

# Sort links by ascending p-value (smallest p-values first)
sorted_links = sorted(links_list, key=lambda x: x['p_value'])

# Select top 4 links with lowest p-values
top_4_links = sorted_links[:4]

# Print top 4 links
for i, link in enumerate(top_4_links, 1):
    print(f"Top {i}: {link['source_name']} (idx {link['source']}) "
          f"-> {link['target_name']} (idx {link['target']}) "
          f"lag {link['lag']}, p-value={link['p_value']:.5g}, effect={link['effect']:.3f}")


Top 1: x0_t4 (idx 12) -> x0_t0 (idx 20) lag 0, p-value=2.6987e-10, effect=0.200
Top 2: x0_t0 (idx 20) -> x0_t4 (idx 12) lag 0, p-value=2.6987e-10, effect=0.200
Top 3: x2_t4 (idx 14) -> x2_t0 (idx 22) lag 0, p-value=3.6952e-08, effect=0.174
Top 4: x2_t0 (idx 22) -> x2_t4 (idx 14) lag 0, p-value=3.6952e-08, effect=0.174


In [111]:
np.set_printoptions(threshold=np.inf)
print(results['graph'])

[[['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']]

 [['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' '' '' '' '' '']
  ['' ''

In [112]:
def interpret_graph(graph_array, var_names):
    n_vars, _, max_lag_plus_1 = graph_array.shape
    links = []
    # lags indexed from 0 to max_lag (usually max_lag+1 in shape)
    # If lag direction convention is lag = -(index), adjust below accordingly
    for lag_idx in range(max_lag_plus_1):
        lag = -lag_idx  # Assuming lag = 0 means current time, negative lags for past
        for source in range(n_vars):
            for target in range(n_vars):
                link_type = graph_array[source, target, lag_idx]
                if link_type:  # Non-empty string means a link
                    links.append({
                        "source_idx": source,
                        "source_name": var_names[source],
                        "target_idx": target,
                        "target_name": var_names[target],
                        "lag": lag,
                        "link_type": link_type
                    })
    return links

# Usage example:
# Print all discovered links:
def print_links(links):
    if not links:
        print("No links found.")
        return
    for link in links:
        print(f"{link['source_name']} (idx {link['source_idx']}) "
                f"--[{link['link_type']}, lag {link['lag']}]-> "
                f"{link['target_name']} (idx {link['target_idx']})")

    # Example call:


In [113]:
links = interpret_graph(graph, var_names)
print_links(links)

x3_t1 (idx 3) --[-->, lag 0]-> x0_t0 (idx 20)
x3_t1 (idx 3) --[-->, lag 0]-> x1_t0 (idx 21)
x0_t3 (idx 8) --[-->, lag 0]-> x1_t0 (idx 21)
x1_t3 (idx 9) --[-->, lag 0]-> x1_t0 (idx 21)
x0_t4 (idx 12) --[-->, lag 0]-> x0_t0 (idx 20)
x0_t4 (idx 12) --[-->, lag 0]-> x1_t0 (idx 21)
x0_t4 (idx 12) --[-->, lag 0]-> x2_t0 (idx 22)
x1_t4 (idx 13) --[-->, lag 0]-> x0_t0 (idx 20)
x1_t4 (idx 13) --[-->, lag 0]-> x1_t0 (idx 21)
x1_t4 (idx 13) --[-->, lag 0]-> x2_t0 (idx 22)
x2_t4 (idx 14) --[-->, lag 0]-> x2_t0 (idx 22)
x2_t4 (idx 14) --[-->, lag 0]-> x3_t0 (idx 23)
x3_t4 (idx 15) --[-->, lag 0]-> x2_t0 (idx 22)
x3_t4 (idx 15) --[-->, lag 0]-> x3_t0 (idx 23)
x3_t5 (idx 19) --[-->, lag 0]-> x0_t0 (idx 20)
x3_t5 (idx 19) --[-->, lag 0]-> x1_t0 (idx 21)
x0_t0 (idx 20) --[<--, lag 0]-> x3_t1 (idx 3)
x0_t0 (idx 20) --[<--, lag 0]-> x0_t4 (idx 12)
x0_t0 (idx 20) --[<--, lag 0]-> x1_t4 (idx 13)
x0_t0 (idx 20) --[<--, lag 0]-> x3_t5 (idx 19)
x1_t0 (idx 21) --[<--, lag 0]-> x3_t1 (idx 3)
x1_t0 (idx 21) --[<